# Problem Set 5: Constraint Satisfaction Problems (CSPs)

In this pset, you will:

- Implement a CSP solver with AC-3 and MRV heuristics from scratch and analyze its efficiency w/o these improvements
- Apply CSP on course scheduling

## 1. Backtracking with AC-3 and MRV

In this section, you will implement the core CSP solver including backtracking search, AC-3 arc consistency, and the variable selection heuristics. Start by reviewing the provided base classes and interfaces. Then, follow the instructions to complete the implementation. Your solver will be tested on some simple problems to verify correctness and demonstrate practical usage.

**Key data structures:**

- **variable:** `var` Any hashable identifier representing a CSP variable.
- **domain:** `dom` A dictionary mapping each variable to its list of possible values.
- **assignment:** `asg` A dictionary mapping each variable to its assigned value.
- **neighbor:** `neigh` A dictionary mapping each variable to the set of variables it shares binary constraints with.

Hint: If you find the running time unreasonably long, check:

- Are you looping over all constraints frequently instead of retrieving relevant ones using specific data structures?
- Are you adding redundant constraints that are always satisfied?
- Are you updating the domains correctly during search?
- Are you stuffing AC-3 with redundant initial arcs?

### Base CSP Class

In [1]:
from collections import deque, defaultdict
# deque is a double-ended queue that allows appending and popping from both ends in O(1) time
# we use it instead of list where a queue is needed to speed up the implementation

class BinaryConstraint:
    """
    For simplicity, we only consider binary constraints in our CSP implementation
    You need to identify all binary constraints and only use this class for them
    Note: pred() is the function determining whether the constraint is satisfied
    """
    def __init__(self, var1, var2, pred):
        self.var1 = var1
        self.var2 = var2
        self.pred = pred

    def involves(self, var):
        return var == self.var1 or var == self.var2
    
    def satisfied(self, asg):
        """ Check if the constraint is satisfied under the given assignment asg (a dict) """
        # If either variable is not assigned yet, we consider the constraint satisfied
        if self.var1 not in asg or self.var2 not in asg:
            return True
        return self.pred(asg[self.var1], asg[self.var2])
    

class CSP:
    """
    The CSP solver class
    Access neighbors and constraints through csp.neigh and csp.con_map ({(var1, var2): [list of binary constraints involving var1 and var2]})
    Refer to csp.add_constraint() for how con_map is constructed
    csp.init_dom is the initial domain for each variable, while csp.dom is the working domain during search
    AC-3 and backtracking search can directly use and modify csp.dom
    When starting a new search, csp.init() is called to reset csp.dom
    """
    def __init__(self, vars=None, doms=None, cons=None):
        self.vars = []
        self.init_dom = {}
        self.cons = []
        self.neigh = defaultdict(set)
        self.con_map: dict[tuple, list[BinaryConstraint]] = defaultdict(list)
        self.dom = {}
        self.init(doms=doms, cons=cons)

    def add_variable(self, var, dom):
        assert var not in self.dom
        self.vars.append(var)
        self.init_dom[var] = list(dom)

    def add_constraint(self, c: BinaryConstraint):
        self.cons.append(c)
        self.neigh[c.var1].add(c.var2)
        self.neigh[c.var2].add(c.var1)
        self.con_map[(c.var1, c.var2)].append(c)
        self.con_map[(c.var2, c.var1)].append(c)

    def init(self, doms=None, cons=None):
        if doms:
            for v, d in doms.items():
                self.add_variable(v, d)
        if cons:
            for c in cons:
                self.add_constraint(c)
        self.dom = {v: list(d) for v, d in self.init_dom.items()}  # reset working domain

    def consistent(self, var, val, assignment):
        """
        Check if the assignment of val to var is consistent with the current assignment
        We assume that upon assigning val to var, all other assignments do not break any constraints
        Therefore, we only need to do incremental checking in an efficient way
        """
        tmp = {**assignment, var: val}
        for nb in self.neigh[var]:
            for c in self.con_map[(var, nb)]:
                if not c.satisfied(tmp):
                    return False
        return True

### 1.2 AC-3 (6pt)
Implement AC-3

In [2]:
def remove_inconsistent_values(Xi, Xj, csp: CSP):
    """
    Remove values from Xi's domain that no Xj's value can satisfy their constraints
    Hint: refer to the CSP implementation for how to access constraints between Xi and Xj
    Hint: unlike the pseudocode, this function also needs to further return the list of removed Xi values
    Return: (removed: bool, pruned_vals: list)
    """
    ### YOUR CODE HERE ###
    removed = False
    pruned_vals = []
    constraints = csp.con_map.get((Xi, Xj), []) ## get constraints if there is, otherwise empty
    
    if not constraints:
        return False, []
    
    values_to_remove = []
    for xi_val in csp.dom[Xi]:
        has_support = False
        for xj_val in csp.dom[Xj]:
            temp_assignment = {Xi: xi_val, Xj: xj_val}
            # Check if this combination satisfies all constraints between Xi and Xj
            all_satisfied = all(
                constraint.satisfied({Xi: xi_val, Xj: xj_val})
                for constraint in constraints
            )
            
            if all_satisfied:
                has_support = True
                break
        
        if not has_support:
            values_to_remove.append(xi_val)
            pruned_vals.append(xi_val)
            removed = True
    
    for val in values_to_remove:
        csp.dom[Xi].remove(val)
    
    return removed, pruned_vals
    
    
    


def AC3(csp: CSP, initial_arcs=None):
    """
    AC-3 algorithm for constraint propagation
    Update csp.dom in place to the pruned domain if consistent
    Since AC-3 is both a preprocessing step and can be used during search, we allow initial_arcs to be provided
    Also note that for backtracking, we need to keep track of all pruned values, so this function additionally returns the pruned values

    initial_arcs: if provided, start this list of arcs; 
                  otherwise, start from all related pairs of variables from the csp in both directions
    return: (consistent_flag, pruned_vals), 
             where consistent_flag indicates whether the CSP is still consistent after AC-3 and
                    pruned_vals are pruned values ({var: pruned_values}) in dom
    """
    arcs = deque()
    for arc in initial_arcs or csp.con_map:
        arcs.append(arc)
    pruned = defaultdict(list)
    ### YOUR CODE HERE ###

    while arcs:
        arc = arcs.popleft()
        removed = remove_inconsistent_values(arc[0], arc[1], csp)
        if removed[0]:
            pruned[arc[0]].extend(removed[1])

            if not csp.dom[arc[0]]:
                return False, dict(pruned)

            for i in csp.neigh[arc[0]]:
                if i != arc[1]:
                    arcs.append((i, arc[0]))

    return True, dict(pruned)

In [3]:
## --- IGNORE --- ##

# from sudoku import Sudoku
# from time import time

# puzzle = Sudoku(3, seed=42).difficulty(0.6)

# vars = [(r, c) for r in range(9) for c in range(9)]
# doms = {v: [puzzle.board[v[0]][v[1]]] if puzzle.board[v[0]][v[1]] else list(range(1, 10)) for v in vars}

# def all_diff(vars):
#     for i in range(len(vars)):
#         for j in range(i + 1, len(vars)):
#             yield BinaryConstraint(vars[i], vars[j], lambda x, y: x != y)

# cons = []
# for r in range(9):
#     row_vars = [(r, c) for c in range(9)]
#     cons.extend(all_diff(row_vars))
# for c in range(9):
#     col_vars = [(r, c) for r in range(9)]
#     cons.extend(all_diff(col_vars))
# for br in range(0, 9, 3):
#     for bc in range(0, 9, 3):
#         box_vars = [(r, c) for r in range(br, br + 3) for c in range(bc, bc + 3)]
#         cons.extend(all_diff(box_vars))

# cons_dict = {i.var1:i.var2 for i in cons}
# cons_dict

### Q1.3 Backtracking (8 pts)
Implement backtracking

In [4]:
## --- IGNORE --- ##
# mydict = {'a':[1,2,3], 'b': [3,2,4,5], 'c': [2,5,78,3], 'd': [1,5,7,8,4]}
# assignment = {'a': 1, 'b': 2}
# print(mydict)
# filtered_dict = {k:v for k, v in mydict.items() if k not in assignment}
# sorted_dict = sorted(filtered_dict.items(), key=lambda item: len(item[1]))
# sorted_dict[0][0]

In [5]:
import random 
def select_unassigned_variable(assignment, csp: CSP, use_mrv):
    """
    select a value with a choice of Minimum Remaining Values heuristic
    Return the variable with the smallest domain size among unassigned variables
    (Optional) If there is a tie, return the one with the most unassigned neighbors
    """
    ### YOUR CODE HERE ###
    ## assignment {var1: number2, var2: number2, etc} csp.dom {var: list[values]}
    filtered_dom = {k:v for k,v in csp.dom.items() if k not in assignment} # get un assigned vars
    if not filtered_dom:
        return None
    if use_mrv:
        sorted_dom = sorted(filtered_dom.items(), key=lambda item: len(item[1]))
        return sorted_dom[0][0]
    else:
        return random.choice(list(filtered_dom))



def recursive_backtracking(assignment, csp: CSP, use_ac3=False, use_mrv=False):
    """
    Solve the CSP using backtracking search with options to use AC-3 and/or MRV
    Hint: During backtracking, AC-3 starts from arcs between the newly assigned variable and its unassigned neighbors
    Hint: You need to restore the pruned values (if using AC-3) before evaluating the next value
    Return: a satisfying assignment as a dict {var: value}, or None if no solution exists
    """
    ### YOUR CODE HERE ###
    if (len(assignment) == len(csp.vars)):
        return assignment
    var = select_unassigned_variable(assignment, csp, use_mrv)
    for value in csp.dom[var]:
        # if value is constitent with assignment given constraints csp
        # restore the pruned values (if using AC-3) before evaluating the next value
        if csp.consistent(var, value, assignment):
            assignment[var] = value
            if use_ac3:
                curr_domain = {k:list(v) for k,v in csp.dom.items()}

                csp.dom[var] = [value]
                
                arcs = [(neighbor, var) for neighbor in csp.neigh[var] if neighbor not in assignment]
                ok, _ = AC3(csp, arcs)
                if not ok:
                    csp.dom = curr_domain
                    # csp.dom[var]
                    assignment.pop(var)
                    continue
            result = recursive_backtracking(assignment, csp, use_ac3, use_mrv)
            if result is not None:
                return result
            if use_ac3:
                csp.dom = curr_domain
            assignment.pop(var)
    return None
    


def backtracking_search(csp: CSP, use_ac3=False, use_mrv=False):
    asg = {}
    csp.init()
    if use_ac3:
        ok, _ = AC3(csp)
        if not ok:
            return None
    result = recursive_backtracking(asg, csp, use_ac3, use_mrv)
    return result

## 2. Sudoku

Below is a toy Sudoku test example. If your implementation is correct, the completed Sudoku grid will be displayed. The provided code illustrates how to construct the CSP using the previously defined APIs.

In [6]:
pip install py-sudoku

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\rashd\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
from sudoku import Sudoku
from time import time

puzzle = Sudoku(3, seed=42).difficulty(0.6)
puzzle.show()

Puzzle has multiple solutions
+-------+-------+-------+
| 5     | 4 6 9 | 2     |
| 2   1 |   3   |     4 |
|     4 | 1 2   | 9 8   |
+-------+-------+-------+
| 8 1 6 |       |     7 |
|     2 |       |     9 |
|       | 7   1 |     8 |
+-------+-------+-------+
| 4     | 3     | 5     |
|   3   |       | 8     |
|   2   | 8   5 | 4 1   |
+-------+-------+-------+



### Building the CSP

In [8]:
vars = [(r, c) for r in range(9) for c in range(9)]
doms = {v: [puzzle.board[v[0]][v[1]]] if puzzle.board[v[0]][v[1]] else list(range(1, 10)) for v in vars}

def all_diff(vars):
    for i in range(len(vars)):
        for j in range(i + 1, len(vars)):
            yield BinaryConstraint(vars[i], vars[j], lambda x, y: x != y)

cons = []
for r in range(9):
    row_vars = [(r, c) for c in range(9)]
    cons.extend(all_diff(row_vars))
for c in range(9):
    col_vars = [(r, c) for r in range(9)]
    cons.extend(all_diff(col_vars))
for br in range(0, 9, 3):
    for bc in range(0, 9, 3):
        box_vars = [(r, c) for r in range(br, br + 3) for c in range(bc, bc + 3)]
        cons.extend(all_diff(box_vars))

sudoku_csp = CSP(vars, doms, cons)

### Q2.1 Experiments and Analysis (3 pts)
Run and analyze the following 3 settings.

In [9]:
s = time()
sol = backtracking_search(sudoku_csp)
print("Naive backtracking:", round(time() - s, 4)) 
for (r, c), v in sol.items():
    if not puzzle.board[r][c]:
        puzzle.board[r][c] = v
puzzle.show()

KeyboardInterrupt: 

In [10]:
s = time()
sol = backtracking_search(sudoku_csp, use_ac3=True)
print("Time with AC-3:", round(time() - s, 4)) 
for (r, c), v in sol.items():
    if not puzzle.board[r][c]:
        puzzle.board[r][c] = v
puzzle.show()

Time with AC-3: 0.1425
Puzzle has multiple solutions
+-------+-------+-------+
| 5 7 8 | 4 6 9 | 2 3 1 |
| 2 9 1 | 5 3 8 | 7 6 4 |
| 3 6 4 | 1 2 7 | 9 8 5 |
+-------+-------+-------+
| 8 1 6 | 9 5 2 | 3 4 7 |
| 7 4 2 | 6 8 3 | 1 5 9 |
| 9 5 3 | 7 4 1 | 6 2 8 |
+-------+-------+-------+
| 4 8 9 | 3 1 6 | 5 7 2 |
| 1 3 5 | 2 7 4 | 8 9 6 |
| 6 2 7 | 8 9 5 | 4 1 3 |
+-------+-------+-------+



In [11]:
sudoku_csp.init()
s = time()
sol = backtracking_search(sudoku_csp, use_ac3=True, use_mrv=True) 
print("Time with AC-3 and MRV:", round(time() - s, 4))
for (r, c), v in sol.items():
    if not puzzle.board[r][c]:
        puzzle.board[r][c] = v
puzzle.show()

Time with AC-3 and MRV: 0.0677
Puzzle has multiple solutions
+-------+-------+-------+
| 5 7 8 | 4 6 9 | 2 3 1 |
| 2 9 1 | 5 3 8 | 7 6 4 |
| 3 6 4 | 1 2 7 | 9 8 5 |
+-------+-------+-------+
| 8 1 6 | 9 5 2 | 3 4 7 |
| 7 4 2 | 6 8 3 | 1 5 9 |
| 9 5 3 | 7 4 1 | 6 2 8 |
+-------+-------+-------+
| 4 8 9 | 3 1 6 | 5 7 2 |
| 1 3 5 | 2 7 4 | 8 9 6 |
| 6 2 7 | 8 9 5 | 4 1 3 |
+-------+-------+-------+



What is the difference between the three runs? Why does these changes work particularly well with Sudoku?

**YOUR ANSWER:**

**Difference between the three runs:**

1. **Naive backtracking**: This approach uses simple backtracking without any optimization. On my side, it did not run at all, I waited for it for about an hour and then interrupted it. I assume this is because the search space is enormous, and without any constraint propagation or heuristics, the algorithm explores many paths before finding a solution.

2. **Backtracking with AC-3**: AC-3 (Arc Consistency 3) performs constraint propagation to reduce domain sizes before and during search. When a variable is assigned, AC-3 removes inconsistent values from neighboring variables' domains. This  reduces the search space by a lot. I took 0.15 seconds to solve the Sudoku puzzle using this method.

3. **Backtracking with AC-3 and MRV**: The Minimum Remaining Values (MRV) heuristic always selects the variable with the smallest remaining domain size first because it is more likely to fail. This makes it detect conflicts earlier. Combined with AC-3, this typically provides the best performance, which is 0.12 seconds.

These changes work particularly well with Sudoku because:
- Sudoku has a high degree of constraint between variables (cells), so constraint propagation (AC-3) effectively reduces the search space.
- The MRV heuristic helps to quickly identify and resolve the most constrained cells, which often leads to faster solutions.

## 3. Course Scheduling

This exercise extends our CSP solver to handle backtracking for course scheduling.

### Q3.1 Formulation (3 pts)

Evelyn, a first-year CS MSE student, is planning her coursework to graduate within three semesters.  
She needs to select a total of 8 courses (Fall 25: 3, Spring 26: 3, Fall 26: 2), subject to the following constraints:

- **Semester limit:** The course she chooses must be available in that semester. This unary constraint should be represented as the domain.

- **No time overlap:** No two courses taken in the same semester may overlap in meeting time.

- **No duplicate courses:** Evelyn is confident that she won't fail, so a course should not be chosen more than once.

The goal is to find a **valid assignment of 8 courses** that satisfies all constraints.

Assume the set of all possible courses in each semester is $C_i,$ $i=1,2,3$, and the meeting time for each course $c$ is $t(c)$. Please formulate this scenario as a CSP.

**YOUR ANSWER:**


- **Objective:** Find an assignment of values to the variables $X$ such that all constraints are satisfied, resulting in a valid course schedule for Evelyn.
- **Variables:** Let $X = \{F_1, F_2, F_3, S_1, S_2, S_3, F_1^{'}, F_2^{'}\}$ represent the 8 courses Evelyn needs to take. Each variable corresponds to a specific course slot in her schedule.
- **Domains:** The domain for each variable $x_i$ is defined based on the semester:
  - Dom($F_1$) = $C_1$ (Fall 25)
  - Dom($F_2$) = $C_1$ (Fall 25)
  - Dom($F_3$) = $C_1$ (Fall 25)
  - Dom($S_1$) = $C_2$ (Spring 26)
  - Dom($S_2$) = $C_2$ (Spring 26)
  - Dom($S_3$) = $C_2$ (Spring 26)
  - Dom($F_1^{'}$) = $C_3$ (Fall 26)
  - Dom($F_2^{'}$) = $C_3$ (Fall 26)
- **Constraints:**
  1. **No time overlap:** 
      - NoOverlap(t($F_i$), t($F_j$)) (Fall 25)
      - NoOverlap(t($S_i$), t($S_j$)) (Spring 26)
      - NoOverlap(t($F_1^{'}$), t($F_2^{'}$)) (Fall 26)
  2. **No duplicate courses:** for any two variables $X$ and $Y$ in {$F_1, F_2, F_3, S_1, S_2, S_3, F_1^{'}, F_2^{'}$}, the constraint $X \neq Y$ must hold.

- **Binary Constraints:** The binary constraints are defined based on the "No time overlap" and "No duplicate courses" conditions. For each pair of variables that represent courses in the same semester, we add a binary constraint to ensure that their meeting times do not overlap. Additionally, for every pair of variables across all semesters, we add a binary constraint to ensure that no two variables can take the same course.

### Preparation: Load the course list
Reading the course list from "courselist.csv"

In [12]:
import csv

SEMESTERS = ["Fall25", "Spring26", "Fall26"]
SEM_KIND = {"Fall25": "Fall", "Spring26": "Spring", "Fall26": "Fall"}

COURSE_CSV = "courselist.csv"

courses = {}  # {course_id: {"title": str, "area": str, "days": str, "time": str, "semester": set(str)}}

with open(COURSE_CSV, "r", encoding="utf-8") as f:
    r = csv.DictReader(f)
    for row in r:
        cid = row["course_id"].strip()
        title = row.get("title", "").strip()
        area = row.get("core_area", "").strip()
        days = row.get("days", "").strip()
        time_slot = row.get("time", "").strip()
        sems = row.get("semesters", "").strip()  # "Fall" / "Spring" / "Both"

        if sems.lower() == "both":
            offered = {"Fall", "Spring"}
        else:
            offered = {sems.strip()}
        
        courses[cid] = {
            "title": title,
            "area": area,
            "days": days,
            "time": time_slot,
            "semester": offered,
        }

In [13]:
Fall_courses = {course: info for course, info in courses.items() if "Fall" in info["semester"]}
Spring_courses = {course: info for course, info in courses.items() if "Spring" in info["semester"]}

### Q3.2 Building the CSP (5 pts)
Define domains and constraints and initialize the CSP as course_csp 

In [14]:
INDEX_TO_SEM = {
    0: "Fall25", 1: "Fall25", 2: "Fall25",
    3: "Spring26", 4: "Spring26", 5: "Spring26",
    6: "Fall26", 7: "Fall26",
}

vars = list(range(8))

### YOUR CODE HERE to instantiate all the binary constraints and initialize the CSP as course_csp ###
domains = {i: Fall_courses if INDEX_TO_SEM[i].startswith("Fall") else Spring_courses for i in vars}


def parse_days(day_str):
    days = []
    i = 0
    while i < len(day_str):
        if day_str[i:i+2] == "Th":    # Check 2-letter day first
            days.append("Th")         # or "Thursday"
            i += 2
        else:
            days.append(day_str[i])   # Single-letter day
            i += 1
    return set(days)

def has_time_conflict(course_id1, course_id2):
    """Check if two courses have overlapping meeting times"""
    c1 = courses[course_id1]
    c2 = courses[course_id2]
    
    days1 = parse_days(c1["days"])
    days2 = parse_days(c2["days"])

    if not days1.intersection(days2):
        return False  # No overlapping days
    
    # If their times are the same and they share days, they conflict
    return c1["time"] == c2["time"]



cons = []
for i in range(len(vars)):
    for j in range(i + 1, len(vars)):
        cons.append(BinaryConstraint(vars[i], vars[j], lambda x, y: x != y))

for sem in SEMESTERS:
    # Get all variables in this semester
    sem_vars = [v for v in vars if INDEX_TO_SEM[v] == sem]
    
    # Add time conflict constraints between pairs in same semester
    for i in range(len(sem_vars)):
        for j in range(i + 1, len(sem_vars)):
            var1, var2 = sem_vars[i], sem_vars[j]
            cons.append(BinaryConstraint(
                var1, var2, 
                lambda x, y: not has_time_conflict(x, y)
            ))

# course_csp = CSP(vars, all_dif(vars))
course_csp = CSP(vars, domains, cons)

In [15]:
s = time()
solution = backtracking_search(course_csp, use_ac3=True, use_mrv=True)
print(f"Solved in {time() - s} seconds")  # Expected within 0.1 seconds
if solution:
    last_sem = None
    for v in range(8):
        cid = solution[v]
        sem = INDEX_TO_SEM[v]
        if sem != last_sem:
            print("-" * 40)
            print(f"{sem} ({SEM_KIND[sem]}):")
            last_sem = sem
        course = courses[cid]
        print(f"  {cid} | {course['title']} | {course['area']} | {course['days']} {course['time']}")
else:
    print("No feasible schedule found.")

Solved in 0.012276172637939453 seconds
----------------------------------------
Fall25 (Fall):
  601.617 | Distributed Systems | Systems | MW 3
  601.628 | Compilers and Interpreters | Software | MW 4:30
  601.654 | [Medical] Augmented Reality | Applications | TTh 9
----------------------------------------
Spring26 (Spring):
  601.615 | Databases (formerly Database Systems) | Software | MW 10
  601.618 | Operating Systems | Systems | TTh 9
  601.620 | Parallel Programming/Parallel Computing for Data Science | Systems | MW 1:30
----------------------------------------
Fall26 (Fall):
  601.653 | Applications of Augmented Reality | Applications | MW 3
  601.655 | Computer Integrated Surgery I | Applications | MW 1:30


### 3.3 (OPTIONAL) Coverage Constraint

We now impose another constraint to make the setting more realistic. 

There are five core areas: *Applications, Systems, Software, Theory, Reasoning.*  Evelyn must take at least one course from each core area.

However, such constraint cannot be represented directly as binary constraints. Therefore, we introduce auxiliary variables and chains of constraints to encode them. These auxiliary variables are assignable and carry specific logical meanings, but they do not significantly increase the overall complexity when implemented properly, since once one variable is assigned, the arc propagation quickly prunes the domains of the entire chain.

This section is totally optional. Please feel free to go through it if interested.

In [16]:
AREAS_REQUIRED = {"Applications", "Systems", "Software", "Theory", "Reasoning"}

def add_or_constraint(name, vars, csp: CSP):
    """
    This function introduces auxiliary variables to represent the OR relationship among a list of variables
    We are using this function to reduce this relationship: any(load)
    csp: the CSP instance
    name: a unique name for this OR constraint
    vars: a list of variables involved in this OR constraint
    """
    o_prev = f"O_{name}"
    csp.add_variable(o_prev, [0])
    if not vars: return o_prev
    for var in vars:
        o_curr = f"O_{name}_{var}"
        csp.add_variable(o_curr, [0, 1])
        p = f"P_{name}_{var}"
        csp.add_variable(p, [(0, 0), (0, 1), (1, 0), (1, 1)])
        csp.add_constraint(BinaryConstraint(o_prev, p, lambda x, y: x == y[0]))  # how to connect o_prev, o_curr and current var?
        csp.add_constraint(BinaryConstraint(var, p, lambda x, y: x == y[1]))  # i.e. o_curr = o_prev OR (var > 0) 
        csp.add_constraint(BinaryConstraint(o_curr, p, lambda x, y: x == (y[0] or y[1])))
        o_prev = o_curr
    csp.init_dom[o_prev] = [1]  # FORCE THE FINAL OUTPUT TO BE 1


def add_coverage_constraints(csp: CSP):
    """
    Add core area coverage constraints to the CSP
    Hint: Add auxiliary variables fulfill_a_c indicating whether the assignment of course c covers area a
    csp: the WeightedCSP instance
    """
    ### YOUR CODE HERE
    for a in AREAS_REQUIRED:
        fulfill_vars = []
        for c in vars:
            fulfill_var = f"fulfill_{a}_{c}"
            csp.add_variable(fulfill_var, [0, 1])
            csp.add_constraint(BinaryConstraint(c, fulfill_var, lambda x, y, area=a: (courses[x]["area"] == area) == (y == 1)))
            fulfill_vars.append(fulfill_var)
        add_or_constraint(f"coverage_{a}", fulfill_vars, csp)

course_csp = CSP(vars=vars, doms=doms, cons=cons)
add_coverage_constraints(course_csp)

In [17]:
s = time()
solution = backtracking_search(course_csp, use_ac3=True, use_mrv=True)
print(f"Solved in {time() - s} seconds")  # Expected within 0.1 seconds
if solution:
    last_sem = None
    for v in range(8):
        cid = solution[v]
        sem = INDEX_TO_SEM[v]
        if sem != last_sem:
            print("-" * 40)
            print(f"{sem} ({SEM_KIND[sem]}):")
            last_sem = sem
        course = courses[cid]
        print(f"  {cid} | {course['title']} | {course['area']} | {course['days']} {course['time']}")
else:
    print("No feasible schedule found.")

KeyError: 0

Great work 🎉 You’ve successfully completed all CSP-related implementation tasks. Keep up the good progress!
Before submitting, please make sure all code blocks produce the expected outputs, then export your notebook as a PDF and submit it.